In [0]:

# ============================================================
# FACT_ORDER — STEP 1
# CONFIGURATION
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
import uuid

# ============================================================
# PROJECT CONFIGURATION
# Replace placeholders with your environment-specific values.
# ============================================================

PROJECT_ID = "YOUR_GCP_PROJECT_ID"

# Raw Delta
RAW_PATH = "gs://YOUR_RAW_BUCKET_NAME/orders/"

# BigQuery
BQ_DATASET = "Gcp_Curated"

TARGET_TABLE = (
    f"{PROJECT_ID}.{BQ_DATASET}.FACT_ORDER"
)

STAGE_TABLE = (
    f"{PROJECT_ID}.{BQ_DATASET}.FACT_ORDER_STAGE"
)

# Existing dimension
DIM_CUSTOMER_TABLE = (
    f"{PROJECT_ID}.{BQ_DATASET}.DIM_CUSTOMER"
)

# Temporary bucket for BigQuery connector
BQ_TEMP_BUCKET = "gs://YOUR_BQ_TEMP_BUCKET/"

# Watermark
WATERMARK_TABLE = "YOUR_CATALOG.YOUR_WATERMARK_TABLE"

# Pipeline metadata
PIPELINE_NAME = "fact_order_pipeline"
SOURCE_SYSTEM = "CloudSQL"

# Generate a unique pipeline run ID
RUN_ID = str(uuid.uuid4())

print("FACT_ORDER pipeline initialized")
print("RUN_ID:", RUN_ID)


In [0]:
# ============================================================
# FACT_ORDER — STEP 2
# READ RAW ORDERS
# ============================================================

raw_orders = (
    spark.read
    .format("delta")
    .load(RAW_PATH)
)

print("Raw FACT_ORDER source schema:")
raw_orders.printSchema()

print("Total raw records:", raw_orders.count())

display(
    raw_orders
    .orderBy("order_id")
)

In [0]:
# ============================================================
# FACT_ORDER — STEP 3
# GET LAST SUCCESSFUL WATERMARK
# ============================================================

watermark_df = (
    spark.table(WATERMARK_TABLE)
    .filter(F.col("table_name") == "FACT_ORDER")
    .filter(F.col("status") == "SUCCESS")
)

lwv = watermark_df.select("lwv").first()["lwv"]

print("FACT_ORDER LWV:", lwv)

In [0]:
# ============================================================
# FACT_ORDER — STEP 4
# INCREMENTAL EXTRACTION
# ============================================================

order_incremental = (
    raw_orders
    .filter(
        F.col("updated_at") > F.lit(lwv)
    )
)

print("Total raw records :", raw_orders.count())
print("Incremental records:", order_incremental.count())

display(
    order_incremental
    .orderBy("order_id")
)

In [0]:
# ============================================================
# FACT_ORDER — STEP 5
# EXACT DUPLICATE REMOVAL
# ============================================================

before_dedup = order_incremental.count()

order_incremental = (
    order_incremental
    .dropDuplicates()
)

after_dedup = order_incremental.count()

print("Records before exact dedup :", before_dedup)
print("Records after exact dedup  :", after_dedup)
print("Exact duplicates removed   :", before_dedup - after_dedup)

display(
    order_incremental
    .orderBy("order_id")
)

In [0]:
# ============================================================
# FACT_ORDER — STEP 6
# CHECK DUPLICATE ORDER IDs
# ============================================================

order_id_duplicates = (
    order_incremental
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.col("count").desc())
)

print(
    "Number of order_ids having multiple records:",
    order_id_duplicates.count()
)

display(order_id_duplicates)

In [0]:
# ============================================================
# FACT_ORDER — STEP 7
# KEEP LATEST VERSION PER ORDER
# ============================================================

order_window = (
    Window
    .partitionBy("order_id")
    .orderBy(F.col("updated_at").desc())
)

fact_order_latest = (
    order_incremental
    .withColumn("row_num", F.row_number().over(order_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

print("Records after latest-order selection:", fact_order_latest.count())

display(
    fact_order_latest
    .orderBy("order_id")
)

In [0]:
# ============================================================
# FACT_ORDER — STEP 8
# READ DIM_CUSTOMER FROM BIGQUERY
# ============================================================

dim_customer = (
    spark.read
    .format("bigquery")
    .option("table", DIM_CUSTOMER_TABLE)
    .load()
)

print("DIM_CUSTOMER record count:", dim_customer.count())

In [0]:
# ============================================================
# FACT_ORDER — STEP 8.4
# CUSTOMER SCD2 LOOKUP WITH CURRENT-VERSION FALLBACK
# ============================================================

# Historical/as-of match
customer_asof = (
    fact_order_latest.alias("o")
    .join(
        dim_customer.alias("c"),
        (
            (F.col("o.customer_id") == F.col("c.customer_id")) &
            (F.col("o.order_date") >= F.col("c.effective_from")) &
            (F.col("o.order_date") < F.col("c.effective_to"))
        ),
        "left"
    )
    .select(
        F.col("o.*"),
        F.col("c.customer_sk").alias("asof_customer_sk")
    )
)

# Current customer version
current_customer = (
    dim_customer
    .filter(F.col("is_current") == True)
    .select(
        "customer_id",
        F.col("customer_sk").alias("current_customer_sk")
    )
)

# Add current SK as fallback
fact_order_enriched = (
    customer_asof.alias("o")
    .join(
        current_customer.alias("c"),
        F.col("o.customer_id") == F.col("c.customer_id"),
        "left"
    )
    .withColumn(
        "customer_sk",
        F.coalesce(
            F.col("o.asof_customer_sk"),
            F.col("c.current_customer_sk")
        )
    )
    .drop(
        "asof_customer_sk",
        "current_customer_sk"
    )
)

print(
    "FACT_ORDER records:",
    fact_order_enriched.count()
)

print(
    "Records with customer_sk still NULL:",
    fact_order_enriched
        .filter(F.col("customer_sk").isNull())
        .count()
)

display(
    fact_order_enriched
    .orderBy("order_id")
)

In [0]:
# ============================================================
# FACT_ORDER — STEP 9
# ADD CUSTOMER LOOKUP TYPE
# ============================================================

fact_order_enriched = (
    customer_asof.alias("o")
    .join(
        current_customer.alias("c"),
        F.col("o.customer_id") == F.col("c.customer_id"),
        "left"
    )
    .withColumn(
        "customer_sk",
        F.coalesce(
            F.col("o.asof_customer_sk"),
            F.col("c.current_customer_sk")
        )
    )
    .withColumn(
        "customer_lookup_type",
        F.when(
            F.col("o.asof_customer_sk").isNotNull(),
            F.lit("SCD2_ASOF")
        )
        .when(
            F.col("c.current_customer_sk").isNotNull(),
            F.lit("CURRENT_FALLBACK")
        )
        .otherwise(
            F.lit("UNMATCHED")
        )
    )
    .drop(
        "asof_customer_sk",
        "current_customer_sk"
    )
)

display(
    fact_order_enriched
    .groupBy("customer_lookup_type")
    .count()
    .orderBy("customer_lookup_type")
)

In [0]:
# ============================================================
# STEP 8.5 — CHECK FALLBACK USAGE
# ============================================================

fallback_count = (
    customer_asof
    .filter(F.col("asof_customer_sk").isNull())
    .count()
)

print("Orders using current-customer fallback:", fallback_count)

In [0]:
# ============================================================
# FACT_ORDER — STEP 9
# FINAL CUSTOMER ENRICHMENT + ORDER SURROGATE KEY
# ============================================================

fact_order_enriched = (
    customer_asof.alias("o")
    .join(
        current_customer.alias("c"),
        F.col("o.customer_id") == F.col("c.customer_id"),
        "left"
    )
    .select(
        # Keep all columns from ORDER
        *[
            F.col(f"o.{col}")
            for col in fact_order_latest.columns
        ],

        # Historical SCD2 customer SK
        F.col("o.asof_customer_sk"),

        # Current customer SK for fallback
        F.col("c.current_customer_sk")
    )
    .withColumn(
        "customer_sk",
        F.coalesce(
            F.col("asof_customer_sk"),
            F.col("current_customer_sk")
        )
    )
    .withColumn(
        "order_sk",
        F.abs(
            F.xxhash64(
                F.col("order_id").cast("string")
            )
        )
    )
    .drop(
        "asof_customer_sk",
        "current_customer_sk"
    )
)

print("FACT_ORDER records:", fact_order_enriched.count())

display(
    fact_order_enriched
    .select(
        "order_sk",
        "order_id",
        "customer_id",
        "customer_sk",
        "order_date",
        "order_status"
    )
    .orderBy("order_id")
)

In [0]:
# ============================================================
# FACT_ORDER — STEP 10
# PREPARE FACT_ORDER STAGE
# ============================================================

fact_order_stage = (
    fact_order_enriched
    .select(
        "order_sk",
        "order_id",
        "customer_sk",
        "order_date",
        "order_status",
        "shipping_city",
        "shipping_state",
        "total_amount",
        "discount_amount",
        "tax_amount",
        "shipping_amount",
        "created_at",
        "updated_at"
    )
)

print("FACT_ORDER stage records:", fact_order_stage.count())

display(
    fact_order_stage
    .orderBy("order_id")
)

In [0]:
# ============================================================
# STEP 10.1
# VERIFY ONE ROW PER ORDER
# ============================================================

duplicate_orders = (
    fact_order_stage
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate order_ids in stage:",
    duplicate_orders.count()
)

display(duplicate_orders)

In [0]:
# ============================================================
# FACT_ORDER — STEP 11
# WRITE STAGE TABLE TO BIGQUERY
# ============================================================

(
    fact_order_stage
    .write
    .format("bigquery")
    .option("temporaryGcsBucket", BQ_TEMP_BUCKET)
    .mode("overwrite")
    .save(STAGE_TABLE)
)

print("FACT_ORDER_STAGE written successfully.")

In [0]:
# ============================================================
# STEP 11.1 — VERIFY STAGE
# ============================================================

stage_check = (
    spark.read
    .format("bigquery")
    .option("table", STAGE_TABLE)
    .load()
)

print("Stage record count:", stage_check.count())

display(
    stage_check
    .orderBy("order_id")
)

In [0]:
# ============================================================
# FACT_ORDER — STEP 12
# BIGQUERY MERGE
# ============================================================

merge_query = f"""
MERGE `{TARGET_TABLE}` T
USING `{STAGE_TABLE}` S
ON T.order_id = S.order_id

WHEN MATCHED THEN
    UPDATE SET
        customer_sk = S.customer_sk,
        order_date = S.order_date,
        order_status = S.order_status,
        shipping_city = S.shipping_city,
        shipping_state = S.shipping_state,
        total_amount = S.total_amount,
        discount_amount = S.discount_amount,
        tax_amount = S.tax_amount,
        shipping_amount = S.shipping_amount,
        created_at = S.created_at,
        updated_at = S.updated_at,
        dw_updated_at = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN
    INSERT (
        order_sk,
        order_id,
        customer_sk,
        order_date,
        order_status,
        shipping_city,
        shipping_state,
        total_amount,
        discount_amount,
        tax_amount,
        shipping_amount,
        created_at,
        updated_at,
        dw_created_at,
        dw_updated_at
    )
    VALUES (
        S.order_sk,
        S.order_id,
        S.customer_sk,
        S.order_date,
        S.order_status,
        S.shipping_city,
        S.shipping_state,
        S.total_amount,
        S.discount_amount,
        S.tax_amount,
        S.shipping_amount,
        S.created_at,
        S.updated_at,
        CURRENT_TIMESTAMP(),
        CURRENT_TIMESTAMP()
    )
"""

print(merge_query)

In [0]:
# ============================================================
# STEP 12.1 — EXECUTE BIGQUERY MERGE
# ============================================================

from google.auth.transport.requests import Request
import requests

provider = dbutils.credentials.getServiceCredentialsProvider("bigquery")
provider.refresh(Request())

headers = {
    "Authorization": f"Bearer {provider.token}",
    "X-Goog-User-Project": PROJECT_ID,
    "Content-Type": "application/json"
}

url = (
    f"https://bigquery.googleapis.com/bigquery/v2/"
    f"projects/{PROJECT_ID}/queries"
)

body = {
    "query": merge_query,
    "useLegacySql": False,
    "location": "asia-south2"
}

response = requests.post(
    url,
    headers=headers,
    json=body
)

print("HTTP Status:", response.status_code)
print(response.json())

In [0]:
# ============================================================
# FACT_ORDER — STEP 13
# VERIFY TARGET AFTER MERGE
# ============================================================

fact_order_target = (
    spark.read
    .format("bigquery")
    .option("table", TARGET_TABLE)
    .load()
)

print("Total FACT_ORDER records:",
      fact_order_target.count())

print("Distinct order_ids:",
      fact_order_target
      .select("order_id")
      .distinct()
      .count())

print("Distinct order_sk:",
      fact_order_target
      .select("order_sk")
      .distinct()
      .count())

print("Records with NULL customer_sk:",
      fact_order_target
      .filter(F.col("customer_sk").isNull())
      .count())

display(
    fact_order_target
    .orderBy("order_id")
)

In [0]:
# ============================================================
# CHECK STAGE SCHEMA
# ============================================================

stage_check.printSchema()

In [0]:
# ============================================================
# FACT_ORDER — STEP 13
# VERIFY TARGET AFTER MERGE
# ============================================================

fact_order_target = (
    spark.read
    .format("bigquery")
    .option("table", TARGET_TABLE)
    .load()
)

print("Total FACT_ORDER records:",
      fact_order_target.count())

print("Distinct order_ids:",
      fact_order_target
      .select("order_id")
      .distinct()
      .count())

print("Distinct order_sk:",
      fact_order_target
      .select("order_sk")
      .distinct()
      .count())

print("Records with NULL customer_sk:",
      fact_order_target
      .filter(F.col("customer_sk").isNull())
      .count())

display(
    fact_order_target
    .orderBy("order_id")
)

In [0]:
# ============================================================
# FACT_ORDER — STEP 14
# UPDATE WATERMARK AFTER SUCCESSFUL MERGE
# ============================================================

new_lwv = (
    fact_order_stage
    .agg(
        F.max("updated_at").alias("max_updated_at")
    )
    .first()["max_updated_at"]
)

print("New FACT_ORDER LWV:", new_lwv)

if new_lwv is not None:

    spark.sql(f"""
        UPDATE {WATERMARK_TABLE}
        SET
            lwv = TIMESTAMP('{new_lwv}'),
            status = 'SUCCESS'
        WHERE table_name = 'FACT_ORDER'
    """)

    print("FACT_ORDER watermark updated successfully.")

else:

    print(
        "No processed records found. "
        "FACT_ORDER watermark was not changed."
    )

In [0]:
# ============================================================
# STEP 14.1 — VERIFY FACT_ORDER WATERMARK
# ============================================================

display(
    spark.table(WATERMARK_TABLE)
    .filter(F.col("table_name") == "FACT_ORDER")
)